# Benchmark EGARCH

In [2]:
import pandas as pd
import numpy as np 

In [3]:
df = pd.read_csv('../data/df_with_2regimes.csv')
df["date"] = pd.to_datetime(df["date"])
df.head()
df.info()
len(df)
df.isna().sum()
df = df.dropna()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3518 entries, 0 to 3517
Data columns (total 37 columns):
 #   Column                                      Non-Null Count  Dtype         
---  ------                                      --------------  -----         
 0   date                                        3518 non-null   datetime64[ns]
 1   log_return                                  3518 non-null   float64       
 2   realized_variance                           3518 non-null   float64       
 3   realized_volatility                         3518 non-null   float64       
 4   gtrend_pct_change                           3518 non-null   float64       
 5   blockchain_diff_log_n_transactions          3518 non-null   float64       
 6   blockchain_diff_log_transaction_fee_usd     3518 non-null   float64       
 7   blockchain_diff_log_n_unique_addresses      3518 non-null   float64       
 8   blockchain_diff_log_transaction_volume_usd  3518 non-null   float64       
 9   log_volu

In [28]:
print(df['log_return'].dtypes)

float64


In [6]:
n = len(df)

train_size = int(n * 0.60)
eval_size  = int(n * 0.25)

train_start_idx = 0
train_end_idx   = train_size

eval_start_idx = train_end_idx
eval_end_idx   = eval_start_idx + eval_size

test_start_idx = eval_end_idx
test_end_idx   = n

train_df = df.iloc[train_start_idx:train_end_idx].copy()
eval_df  = df.iloc[eval_start_idx:eval_end_idx].copy() 
test_df  = df.iloc[test_start_idx:test_end_idx].copy()

print(f"Train set: {len(train_df)} rows")
print(f"Eval set:  {len(eval_df)} rows")
print(f"Test set:  {len(test_df)} rows")

Train set: 2093 rows
Eval set:  872 rows
Test set:  524 rows


In [7]:
def fit_egarch_benchmark(returns, p=1, q=1, o=1, mean="constant", vol="EGARCH", dist="t", min_obs=200):
    returns = pd.Series(returns).dropna().astype(float)

    if len(returns) < min_obs:
        return None

    try:
        am = arch_model(
            returns * 100.0,
            mean=mean,
            vol=vol,
            p=p,
            o=o,
            q=q,
            dist=dist
        )
        res = am.fit(disp="off")
        return res
    except Exception:
        return None


def one_step_variance_forecast(fitted_model, scale=100.0):
    if fitted_model is None:
        return np.nan

    try:
        fcast = fitted_model.forecast(horizon=1, reindex=False)
        var_scaled = fcast.variance.values[-1, 0]
        return var_scaled / (scale ** 2)
    except Exception:
        return np.nan


def rolling_egarch_benchmark_eval(
    df_full,
    start_idx,
    end_idx,
    return_col="log_return",
    realized_var_col="realized_variance",
    date_col="date",
    rolling_window=365,
    p=1,
    q=1,
    o=1,
    mean="constant",
    vol="EGARCH",
    dist="t",
    min_obs=200
):
    results = []

    for t in range(start_idx - 1, end_idx - 1):
        train_start = max(0, t - rolling_window + 1)
        train_end = t + 1

        train_slice = df_full.iloc[train_start:train_end].copy()
        train_returns = train_slice[return_col].dropna()

        next_row = df_full.iloc[t + 1]

        if len(train_returns) < min_obs:
            results.append({
                date_col: next_row[date_col],
                "actual_var": next_row[realized_var_col],
                "var_egarch_bench": np.nan
            })
            continue

        fitted = fit_egarch_benchmark(
            train_returns,
            p=p,
            q=q,
            o=o,
            mean=mean,
            vol=vol,
            dist=dist,
            min_obs=min_obs
        )

        var_forecast = one_step_variance_forecast(fitted)

        results.append({
            date_col: next_row[date_col],
            "actual_var": next_row[realized_var_col],
            "var_egarch_bench": var_forecast
        })

    return pd.DataFrame(results)

In [8]:
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() == 0:
        return np.nan
    return np.sqrt(np.mean((y_true[mask] - y_pred[mask]) ** 2))


def qlike(y_true, y_pred, eps=1e-12):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = np.isfinite(y_true) & np.isfinite(y_pred) & (y_true > 0) & (y_pred > 0)
    if mask.sum() == 0:
        return np.nan

    yt = y_true[mask]
    yp = y_pred[mask]

    return np.mean(np.log(yp + eps) + yt / (yp + eps))


In [9]:
from arch import arch_model
from itertools import product

In [10]:
egarch_grid = {
    "p": [1, 2, 3],
    "q": [1, 2, 3],
    "o": [1]
}

egarch_param_grid = []
for p, q, o in product(
    egarch_grid["p"],
    egarch_grid["q"],
    egarch_grid["o"]
):
    egarch_param_grid.append({
        "p": p,
        "q": q,
        "o": o
    })

garch_results = []

for i, params in enumerate(egarch_param_grid, 1):
    p = params["p"]
    q = params["q"]
    o = params["o"]

    print(f"\n[{i}/{len(egarch_param_grid)}] Testing EGARCH: {params}")

    try:
        res = rolling_egarch_benchmark_eval(
            df_full=df,
            start_idx=eval_start_idx,
            end_idx=eval_end_idx,
            return_col="log_return",
            realized_var_col="realized_variance",
            date_col="date",
            rolling_window=365,
            p=p,
            q=q,
            o=o,
            mean="constant",
            vol="EGARCH",
            dist="t",
            min_obs=200
        )

        pred_col = "var_egarch_bench"   # use this if you renamed it properly

        valid_mask = (
            np.isfinite(res["actual_var"]) &
            np.isfinite(res[pred_col]) &
            (res[pred_col] > 0)
        )

        n_total = len(res)
        n_valid = valid_mask.sum()

        if n_valid == 0:
            eval_rmse = np.nan
            eval_qlike = np.nan
        else:
            eval_rmse = rmse(
                res.loc[valid_mask, "actual_var"],
                res.loc[valid_mask, pred_col]
            )
            eval_qlike = qlike(
                res.loc[valid_mask, "actual_var"],
                res.loc[valid_mask, pred_col]
            )

        print(f"Eval RMSE : {eval_rmse:.6f}" if np.isfinite(eval_rmse) else "Eval RMSE : nan")
        print(f"Eval QLIKE: {eval_qlike:.6f}" if np.isfinite(eval_qlike) else "Eval QLIKE: nan")
        print(f"Total rows    : {n_total}")
        print(f"Valid eval obs: {n_valid}")

        garch_results.append({
            "p": p,
            "q": q,
            "o": o,
            "eval_rmse": eval_rmse,
            "eval_qlike": eval_qlike,
            "n_total": n_total,
            "n_valid": n_valid,
            "results_df": res
        })

    except Exception as e:
        print(f"Failed for p={p}, q={q}, o={o}: {e}")


[1/9] Testing EGARCH: {'p': 1, 'q': 1, 'o': 1}


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limi

Eval RMSE : nan
Eval QLIKE: 67673717.494214
Total rows    : 872
Valid eval obs: 857

[2/9] Testing EGARCH: {'p': 1, 'q': 2, 'o': 1}


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality con

Eval RMSE : nan
Eval QLIKE: 62017657.136396
Total rows    : 872
Valid eval obs: 858

[3/9] Testing EGARCH: {'p': 1, 'q': 3, 'o': 1}


...

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality con

Eval RMSE : nan
Eval QLIKE: 53082151.103960
Total rows    : 872
Valid eval obs: 839

[9/9] Testing EGARCH: {'p': 3, 'q': 3, 'o': 1}


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limi

Eval RMSE : nan
Eval QLIKE: 60498384.934108
Total rows    : 872
Valid eval obs: 840


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/var/folders/2h/gvrjdw1x6t766jgk66w62_tw0000gn/T/ipykernel_2737/851364608.py:7: RuntimeWarning: overflow encountered in square
  return np.sqrt(np.mean((y_true[mask] - y_pred[mask]) ** 2))


In [12]:
egarch_results_df = pd.DataFrame(garch_results).sort_values("eval_qlike")
print(egarch_results_df.head())

   p  q  o  eval_rmse    eval_qlike  n_total  n_valid  \
3  2  1  1        inf  3.056240e+07      872      852   
6  3  1  1        inf  3.533633e+07      872      834   
2  1  3  1        inf  3.673781e+07      872      868   
4  2  2  1        inf  5.073716e+07      872      857   
7  3  2  1        inf  5.308215e+07      872      839   

                                          results_df  
3            date  actual_var  var_egarch_bench
0...  
6            date  actual_var  var_egarch_bench
0...  
2            date  actual_var  var_egarch_bench
0...  
4            date  actual_var  var_egarch_bench
0...  
7            date  actual_var  var_egarch_bench
0...  


In [ ]:
best_egarch_params = egarch_results_df.iloc[0].to_dict()

best_p = int(best_egarch_params["p"])
best_q = int(best_egarch_params["q"])
best_o = int(best_egarch_params["o"])

In [14]:
def fit_egarch_benchmark(
    returns,
    p=1,
    q=1,
    o=1,
    mean="constant",
    vol="EGARCH",
    dist="t",
    min_obs=200
):
    returns = pd.Series(returns).dropna().astype(float)

    if len(returns) < min_obs:
        return None

    try:
        am = arch_model(
            returns * 100.0,
            mean=mean,
            vol=vol,
            p=p,
            o=o,
            q=q,
            dist=dist
        )
        res = am.fit(disp="off")
        return res
    except Exception:
        return None


def one_step_variance_forecast(fitted_model, scale=100.0):
    if fitted_model is None:
        return np.nan

    try:
        fcast = fitted_model.forecast(horizon=1, reindex=False)
        var_scaled = fcast.variance.values[-1, 0]
        return var_scaled / (scale ** 2)
    except Exception:
        return np.nan


def rolling_egarch_benchmark_eval(
    df_full,
    start_idx,
    end_idx,
    return_col="log_return",
    realized_var_col="realized_variance",
    date_col="date",
    rolling_window=365,
    p=1,
    q=1,
    o=1,
    mean="constant",
    vol="EGARCH",
    dist="t",
    min_obs=200,
    debug=False
):
    results = []

    for t in range(start_idx - 1, end_idx - 1):
        train_start = max(0, t - rolling_window + 1)
        train_end = t + 1

        train_slice = df_full.iloc[train_start:train_end].copy()
        train_returns = train_slice[return_col].dropna()
        next_row = df_full.iloc[t + 1]

        row = {
            date_col: next_row[date_col],
            "actual_var": next_row[realized_var_col],
            "var_egarch_bench": np.nan,
            "fit_failed": False,
            "forecast_failed": False,
            "bad_forecast": False,
            "convergence_flag": np.nan,
            "loglik": np.nan,
            "max_abs_return": np.nan,
            "train_std": np.nan,
            "max_cond_var": np.nan,
            "min_cond_var": np.nan,
            "params": None,
        }

        if len(train_returns) < min_obs:
            row["fit_failed"] = True
            results.append(row)
            continue

        row["max_abs_return"] = np.max(np.abs(train_returns))
        row["train_std"] = np.std(train_returns)

        fitted = fit_egarch_benchmark(
            train_returns,
            p=p,
            q=q,
            o=o,
            mean=mean,
            vol=vol,
            dist=dist,
            min_obs=min_obs
        )

        if fitted is None:
            row["fit_failed"] = True
            results.append(row)
            continue

        try:
            row["convergence_flag"] = getattr(fitted, "convergence_flag", np.nan)
            row["loglik"] = getattr(fitted, "loglikelihood", np.nan)
            row["params"] = fitted.params.to_dict() if hasattr(fitted.params, "to_dict") else dict(fitted.params)

            cond_var = (fitted.conditional_volatility ** 2) / (100.0 ** 2)
            row["max_cond_var"] = np.nanmax(cond_var)
            row["min_cond_var"] = np.nanmin(cond_var)
        except Exception:
            pass

        var_forecast = one_step_variance_forecast(fitted)

        if not np.isfinite(var_forecast):
            row["forecast_failed"] = True
        elif var_forecast <= 0:
            row["bad_forecast"] = True
        elif var_forecast > 1:
            row["bad_forecast"] = True

        row["var_egarch_bench"] = var_forecast
        results.append(row)

        if debug and (row["fit_failed"] or row["forecast_failed"] or row["bad_forecast"]):
            print("\n--- DEBUG EGARCH ISSUE ---")
            print(f"date              : {row[date_col]}")
            print(f"p,q,o             : {p},{q},{o}")
            print(f"fit_failed        : {row['fit_failed']}")
            print(f"forecast_failed   : {row['forecast_failed']}")
            print(f"bad_forecast      : {row['bad_forecast']}")
            print(f"convergence_flag  : {row['convergence_flag']}")
            print(f"loglik            : {row['loglik']}")
            print(f"forecast          : {row['var_egarch_bench']}")
            print(f"max_abs_return    : {row['max_abs_return']}")
            print(f"train_std         : {row['train_std']}")
            print(f"max_cond_var      : {row['max_cond_var']}")
            print(f"min_cond_var      : {row['min_cond_var']}")
            print(f"params            : {row['params']}")

    return pd.DataFrame(results)

In [21]:
res_debug = rolling_egarch_benchmark_eval(
    df_full=df,
    start_idx=eval_start_idx,
    end_idx=eval_end_idx,
    return_col="log_return",
    realized_var_col="realized_variance",
    date_col="date",
    rolling_window=365,
    p=1,
    q=1,
    o=1,
    mean="constant",
    vol="EGARCH",
    dist="t",
    min_obs=200,
    debug=True
) 

# convert to df 
res_debug_df = pd.DataFrame(res_debug)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(



--- DEBUG EGARCH ISSUE ---
date              : 2022-04-14 00:00:00
p,q,o             : 1,1,1
fit_failed        : False
forecast_failed   : False
bad_forecast      : True
convergence_flag  : 0
loglik            : -2695.6994130434227
forecast          : 1.5659474217597371e+69
max_abs_return    : 0.154230988594092
train_std         : 0.038934851450166445
max_cond_var      : 4479.831273815906
min_cond_var      : 1.5159226574465277e-07
params            : {'mu': -0.9741226054133653, 'omega': -6.491731010578878, 'alpha[1]': -221.28411828682636, 'gamma[1]': -1087.775824949386, 'beta[1]': 0.0, 'nu': 2.05}


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
Se


--- DEBUG EGARCH ISSUE ---
date              : 2022-04-20 00:00:00
p,q,o             : 1,1,1
fit_failed        : False
forecast_failed   : False
bad_forecast      : True
convergence_flag  : 0
loglik            : -24784.350968381576
forecast          : 1.797693134862273e+304
max_abs_return    : 0.154230988594092
train_std         : 0.03882784916890931
max_cond_var      : 358.6654877350982
min_cond_var      : 5.6944994936488e-10
params            : {'mu': -32933365.590652745, 'omega': 11.923445688481106, 'alpha[1]': -20640283.798893496, 'gamma[1]': -7406953.122655196, 'beta[1]': 1.0, 'nu': 2.05}


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
Se

...

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
Se


--- DEBUG EGARCH ISSUE ---
date              : 2024-08-15 00:00:00
p,q,o             : 1,1,1
fit_failed        : False
forecast_failed   : False
bad_forecast      : True
convergence_flag  : 4
loglik            : -28585.219828064655
forecast          : 1.797693134862273e+304
max_abs_return    : 0.1123520708132801
train_std         : 0.026792775485371442
max_cond_var      : 0.0006337724166267449
min_cond_var      : 2.0113254321101807e-10
params            : {'mu': -2505377.045366342, 'omega': 0.021739214564224847, 'alpha[1]': -15142125.982612902, 'gamma[1]': 9695066.2898938, 'beta[1]': 0.9874406724026961, 'nu': 3.0520011353801544}

--- DEBUG EGARCH ISSUE ---
date              : 2024-08-17 00:00:00
p,q,o             : 1,1,1
fit_failed        : False
forecast_failed   : False
bad_forecast      : True
convergence_flag  : 0
loglik            : -2531.4673580251356
forecast          : 5.902017013434717e+37
max_abs_return    : 0.1123520708132801
train_std         : 0.02651799818314453
max_cond

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
Se


--- DEBUG EGARCH ISSUE ---
date              : 2024-08-23 00:00:00
p,q,o             : 1,1,1
fit_failed        : False
forecast_failed   : False
bad_forecast      : True
convergence_flag  : 9
loglik            : -9492.68077044376
forecast          : 0.0
max_abs_return    : 0.1123520708132801
train_std         : 0.02658731634868548
max_cond_var      : 1976.6835870662076
min_cond_var      : 27.868342773245512
params            : {'mu': 341018.83731252904, 'omega': 11.166038732131604, 'alpha[1]': 91686.6012590069, 'gamma[1]': 57981.581708936676, 'beta[1]': 0.9999999999999961, 'nu': 2.201464434104935}

--- DEBUG EGARCH ISSUE ---
date              : 2024-08-24 00:00:00
p,q,o             : 1,1,1
fit_failed        : False
forecast_failed   : False
bad_forecast      : True
convergence_flag  : 9
loglik            : -4169.8536436408285
forecast          : 1.797693134862273e+304
max_abs_return    : 0.1123520708132801
train_std         : 0.026747284726063823
max_cond_var      : 1943.8118466975461

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/arch/univariate/base.py:768: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(


In [26]:
problem_mask = (
    res_debug_df["fit_failed"] |
    res_debug_df["forecast_failed"] |
    res_debug_df["bad_forecast"] |
    (~np.isfinite(res_debug_df["var_egarch_bench"]))
)

problem_rows = res_debug_df.loc[problem_mask].copy()

problem_rows[[
    "date",
    "actual_var",
    "fit_failed",
    "forecast_failed",
    "bad_forecast",
    "convergence_flag",
    "var_egarch_bench",
    "max_abs_return",
    "train_std",
    "max_cond_var",
    "min_cond_var"
]].head(20)

,date,actual_var,fit_failed,forecast_failed,bad_forecast,convergence_flag,var_egarch_bench,max_abs_return,train_std,max_cond_var,min_cond_var
7,2022-04-14,0.000575,False,False,True,0,1.565947e+69,0.154231,0.038935,4479.831274,1.515923e-07
13,2022-04-20,0.000514,False,False,True,0,1.797693e+304,0.154231,0.038828,358.665488,5.694499e-10
20,2022-04-27,0.000923,False,False,True,9,1.797693e+304,0.154231,0.038495,4362.108719,3.938684e-06
22,2022-04-29,0.000637,False,False,True,0,1.797693e+304,0.154231,0.038520,4755.071548,4.284555e-05
25,2022-05-02,0.000549,False,False,True,0,1.797693e+304,0.154231,0.038358,462.609158,5.661836e-10
26,2022-05-03,0.000178,False,False,True,0,1.797693e+304,0.154231,0.038344,4911.175334,5.683025e-10
28,2022-05-05,0.002126,False,False,True,0,1.797693e+304,0.154231,0.038267,4970.923594,5.694464e-10
33,2022-05-10,0.004399,False,False,True,9,6.583275e+00,0.154231,0.038826,2118.370909,2.912386e-04
34,2022-05-11,0.015349,False,False,True,0,1.797693e+304,0.154231,0.038802,5527.633874,5.641793e-10
39,2022-05-16,0.002309,False,False,True,4,0.000000e+00,0.154231,0.038249,0.003958,5.663508e-10


In [24]:
len(problem_rows)

55

In [25]:
problem_rows.tail(10)

,date,actual_var,var_egarch_bench,fit_failed,forecast_failed,bad_forecast,convergence_flag,loglik,max_abs_return,train_std,max_cond_var,min_cond_var,params
840,2024-07-25,0.000707,4.987778e+02,False,False,True,4,-1736.600797,0.097822,0.025216,1977.031595,1.396186e-07,"{'mu': -9.036481538833074, 'omega': -7.3436784..."
845,2024-07-30,0.000324,1.354965e+18,False,False,True,9,-2519.728003,0.097822,0.025293,1976.655689,2.445283e-03,"{'mu': -1.0604290875496845, 'omega': 3.5376182..."
849,2024-08-03,0.000581,0.000000e+00,False,False,True,4,-29988.241037,0.097822,0.025522,0.000328,1.847076e-10,"{'mu': 200075.32592114137, 'omega': 0.01732336..."
855,2024-08-09,0.000522,1.797693e+304,False,False,True,0,-4171.200499,0.112352,0.026631,1978.480852,1.942549e-10,"{'mu': 0.06597176499751273, 'omega': 1.7392490..."
861,2024-08-15,0.001417,1.797693e+304,False,False,True,4,-28585.219828,0.112352,0.026793,0.000634,2.011325e-10,"{'mu': -2505377.045366342, 'omega': 0.02173921..."
863,2024-08-17,0.000107,5.902017e+37,False,False,True,0,-2531.467358,0.112352,0.026518,1977.255422,7.032117e-08,"{'mu': -2.1971987203850865, 'omega': -7.259862..."
864,2024-08-18,0.000667,1.797693e+304,False,False,True,0,-4129.643883,0.112352,0.026491,1976.689873,1.905635e-10,"{'mu': 0.2383731849198566, 'omega': -3.3784098..."
869,2024-08-23,0.000932,0.000000e+00,False,False,True,9,-9492.680770,0.112352,0.026587,1976.683587,2.786834e+01,"{'mu': 341018.83731252904, 'omega': 11.1660387..."
870,2024-08-24,0.000224,1.797693e+304,False,False,True,9,-4169.853644,0.112352,0.026747,1943.811847,1.993470e-10,"{'mu': 0.288136250039367, 'omega': 1.851676373..."
871,2024-08-25,0.000052,0.000000e+00,False,False,True,9,-7817.159851,0.112352,0.026745,1975.279811,1.952734e-10,"{'mu': 293.1871314719763, 'omega': 11.17786123..."
